# DQ Framework (lightweight PySpark, fail-fast)

Mirror của `databricks_processing/dq/` package. Notebook này self-contained để chạy qua `%run ./dq_framework` trên Databricks Community Edition (không cần file YAML/Repos).

**Giữ đồng bộ** với `dq/rules.py`, `dq/runner.py`, `dq/validations.yml`.

In [ ]:
# DQ framework (mirror của databricks_processing/dq/ — giữ đồng bộ).
# Chạy qua:  %run ./dq_framework   (từ bronze/silver/gold notebook cùng thư mục)
from pyspark.sql.functions import col, count, when, lit


def _check_no_nulls(spark, df, params):
    cols = params["columns"]
    nulls_row = df.select(
        [count(when(col(c).isNull() | (col(c) == ""), c)).alias(c) for c in cols]
    ).collect()[0].asDict()
    bad = {c: int(v) for c, v in nulls_row.items() if v}
    return {"passed": not bad, "failing_count": sum(bad.values()),
            "detail": f"null/empty counts: {bad}" if bad else "no nulls"}


def _check_unique(spark, df, params):
    cols = params["columns"]
    total = df.count()
    distinct = df.select(*cols).distinct().count()
    dups = total - distinct
    return {"passed": dups == 0, "failing_count": dups,
            "detail": f"{dups} duplicate row(s) on {cols} (total={total})"}


def _check_range(spark, df, params):
    c = params["column"]; mn = params.get("min"); mx = params.get("max")
    cond = lit(False)
    if mn is not None: cond = cond | (col(c) < mn)
    if mx is not None: cond = cond | (col(c) > mx)
    out = df.filter(cond).count()
    return {"passed": out == 0, "failing_count": out,
            "detail": f"{out} row(s) outside [{mn}, {mx}] on {c}"}


def _check_row_count_at_least(spark, df, params):
    n = df.count(); mn = params["min"]
    return {"passed": n >= mn, "failing_count": max(0, mn - n),
            "detail": f"{n} rows (min required {mn})"}


def _check_allowed_values(spark, df, params):
    c = params["column"]; allowed = params["values"]
    bad = df.filter(~col(c).isin(*allowed)).count()
    return {"passed": bad == 0, "failing_count": bad,
            "detail": f"{bad} row(s) with {c} not in {allowed}"}


def _check_referential_integrity(spark, df, params):
    child_cols = params["child_columns"]; parent_table = params["parent_table"]
    parent_cols = params["parent_columns"]
    parent = spark.table(parent_table).select(*parent_cols).distinct()
    for pc in parent_cols:
        parent = parent.withColumnRenamed(pc, f"_p_{pc}")
    join_cond = None
    for cc, pc in zip(child_cols, parent_cols):
        clause = col(cc) == col(f"_p_{pc}")
        join_cond = clause if join_cond is None else join_cond & clause
    orphans = df.select(*child_cols).distinct().join(parent, join_cond, "left_anti")
    n = orphans.count()
    return {"passed": n == 0, "failing_count": n,
            "detail": f"{n} orphan child key(s) not in {parent_table}"}


def _check_custom_sql(spark, df, params):
    n = spark.sql(params["sql"]).collect()[0][0]
    return {"passed": n == 0, "failing_count": int(n),
            "detail": f"{n} violating row(s) per custom SQL"}


REGISTRY = {
    "no_nulls": _check_no_nulls, "unique": _check_unique, "range": _check_range,
    "row_count_at_least": _check_row_count_at_least, "allowed_values": _check_allowed_values,
    "referential_integrity": _check_referential_integrity, "custom_sql": _check_custom_sql,
}


In [ ]:
class DQFailure(Exception):
    """Raise khi >=1 rule fail."""
    def __init__(self, results):
        self.results = results
        failed = [r for r in results if not r["passed"]]
        lines = [f"  - [{r['rule']}] {r['detail']} (failing={r['failing_count']})" for r in failed]
        super().__init__("DQ gate FAILED: " + str(len(failed)) + " rule(s) violated.\n" + "\n".join(lines))


def _resolve_df(spark, target):
    return spark.table(target) if isinstance(target, str) else target


def run_gate(spark, target, rule_set=None, rules=None, fail_fast=True):
    """Chạy DQ gate. target = DataFrame hoặc tên bảng. rule_set = key trong DEFAULT_RULES."""
    if rules is None:
        if rule_set is None:
            raise ValueError("can rule_set hoac rules")
        if rule_set not in DEFAULT_RULES:
            raise KeyError("rule_set '" + rule_set + "' khong co. Co: " + str(list(DEFAULT_RULES)))
        rules = DEFAULT_RULES[rule_set]
    df = _resolve_df(spark, target)
    results = []
    for rule in rules:
        rtype = rule["rule"]
        if rtype not in REGISTRY:
            raise KeyError("rule type '" + rtype + "' khong ho tro")
        params = {k: v for k, v in rule.items() if k not in ("rule", "name")}
        name = rule.get("name", rtype)
        res = REGISTRY[rtype](spark, df, params)
        res["rule"] = name
        results.append(res)
        if fail_fast and not res["passed"]:
            raise DQFailure(results)
    failed = [r for r in results if not r["passed"]]
    if failed:
        raise DQFailure(results)
    print("DQ PASS [" + str(rule_set) + "]: " + str(len(results) - len(failed)) + "/" + str(len(results)) + " rules OK")
    return {"rule_set": rule_set, "total": len(results), "passed": len(results) - len(failed), "failed": len(failed)}


In [ ]:
# Rule definitions (mirror databricks_processing/dq/validations.yml — giữ đồng bộ).
DEFAULT_RULES = {
    "bronze": [
        {"rule": "row_count_at_least", "name": "bronze_has_data", "min": 1},
        {"rule": "no_nulls", "name": "bronze_no_null_keys", "columns": ["user_id", "product_id"]},
    ],
    "silver": [
        {"rule": "row_count_at_least", "name": "silver_has_data", "min": 1},
        {"rule": "no_nulls", "name": "silver_no_nulls", "columns": ["user_id", "product_id", "event_time"]},
        {"rule": "unique", "name": "silver_no_dup_row_hash", "columns": ["_row_hash"]},
        {"rule": "range", "name": "silver_price_positive", "column": "price", "min": 0},
        {"rule": "allowed_values", "name": "silver_event_type_domain", "column": "event_type", "values": ["view", "cart", "purchase"]},
    ],
    "dim_product": [
        {"rule": "no_nulls", "columns": ["product_key", "product_id"]},
        {"rule": "unique", "columns": ["product_key"]},
        {"rule": "unique", "name": "product_natural_key_unique", "columns": ["product_id"]},
    ],
    "dim_user": [
        {"rule": "no_nulls", "columns": ["user_key", "user_id", "valid_from"]},
        {"rule": "unique", "columns": ["user_key"]},
        {"rule": "allowed_values", "column": "is_current", "values": [True, False]},
    ],
    "dim_date": [
        {"rule": "no_nulls", "columns": ["date_key"]},
        {"rule": "unique", "columns": ["date_key"]},
    ],
    "fact_daily_performance": [
        {"rule": "row_count_at_least", "min": 1},
        {"rule": "range", "name": "gmv_non_negative", "column": "total_gmv", "min": 0},
        {"rule": "range", "name": "orders_non_negative", "column": "total_orders", "min": 0},
        {"rule": "referential_integrity", "name": "fk_date", "child_columns": ["date_key"], "parent_table": "workspace.gold_cosmetics.dim_date", "parent_columns": ["date_key"]},
    ],
    "fact_user_funnel": [
        {"rule": "row_count_at_least", "min": 1},
        {"rule": "referential_integrity", "name": "fk_product", "child_columns": ["product_key"], "parent_table": "workspace.gold_cosmetics.dim_product", "parent_columns": ["product_key"]},
        {"rule": "referential_integrity", "name": "fk_user", "child_columns": ["user_key"], "parent_table": "workspace.gold_cosmetics.dim_user", "parent_columns": ["user_key"]},
        {"rule": "referential_integrity", "name": "fk_date", "child_columns": ["date_key"], "parent_table": "workspace.gold_cosmetics.dim_date", "parent_columns": ["date_key"]},
    ],
    "fact_rfm_segmentation": [
        {"rule": "row_count_at_least", "min": 1},
        {"rule": "range", "name": "recency_non_negative", "column": "recency", "min": 0},
        {"rule": "range", "name": "frequency_at_least_one", "column": "frequency", "min": 1},
        {"rule": "range", "name": "monetary_non_negative", "column": "monetary", "min": 0},
        {"rule": "allowed_values", "name": "segment_domain", "column": "customer_segment", "values": ["VIP", "Newbie", "Regular", "Churning"]},
        {"rule": "referential_integrity", "name": "fk_user", "child_columns": ["user_key"], "parent_table": "workspace.gold_cosmetics.dim_user", "parent_columns": ["user_key"]},
    ],
}
print("DQ framework loaded. Dung: run_gate(spark, df_hoac_table, rule_set='silver')")
